# 08 — Standalone steering-direction trainer (no generation, trace-only)

**Run on Colab.** Single-purpose notebook: take more rows from the conflict-pairs source,
build matched-baseline S/U prompt pairs with a **shared prefilled response** per item
(so role placement is the only difference), forward-pass the model to capture residuals
at `response_first` and `response_last`, and fit three probe families per
(position × layer):

- **MM** (mass-mean / Marks & Tegmark): `mean(S) − mean(U)` per layer.
- **pca_diff**: PCA(1) on `h_S − h_U` per pair (repeng default).
- **pca_center**: PCA(1) on per-pair-centred points (repeng's `pca_center`).

Output (saved to Drive): `<DRIVE_ROOT>/<model_slug>/exp08_directions/directions.npz`
plus a `manifest.json`. **No generation, no judging, no behavioural eval** — that's all
done downstream in `07_steering_authority.ipynb` (point its `EXP6_DIR` / `PCA_PATH` at
this output to use the new directions).

Improvements over the exp6 trace pipeline:

- Larger source pool (`data/conflict_pairs.jsonl`, 21899 rows, vs the 1k stratified subset).
- Per-row prefilled response is the row's *own* aligned response (`s_aligned_response`
  for s-instruction items, `u_aligned_response` for u-instruction items) — same string
  in both S and U arms, so role placement is the only varying axis.
- Disk caching per item under `OUT_DIR/act_cache/` so a long trace run is resumable.

In [ ]:
# Install + sys.path. Match the convention used in 06_structural_authority.ipynb.
import os, sys, importlib, site
REPO_DIR = "/content/Mech_spoof"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/ChuloIva/Mech_spoof.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull -q
!pip install -q -e {REPO_DIR}
os.environ["MECH_SPOOF_ROOT"] = REPO_DIR
src_path = f"{REPO_DIR}/src"
if src_path not in sys.path:
    sys.path.insert(0, src_path)
importlib.invalidate_caches()
site.main()
import mech_spoof
print("mech_spoof loaded from:", mech_spoof.__file__)

In [ ]:
# Auth + Drive
import os
from google.colab import drive, userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
drive.mount("/content/drive")

In [ ]:
# Config + imports.
from __future__ import annotations

import json
import random
import time
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
from sklearn.decomposition import PCA

from mech_spoof.configs import MODEL_CONFIGS
from mech_spoof.models import load_model

# ---- paths ----
REPO = Path(REPO_DIR)
DRIVE_ROOT = Path("/content/drive/MyDrive/mech_spoof_results")

# ---- knobs ----
MODEL_KEY = "qwen"
SOURCE_PATH = REPO / "data" / "conflict_pairs.jsonl"   # full 21k pool
N_ROWS = 2500                # source rows to sample (each → up to 2 items)
SEED = 42
BASELINE_SYSTEM = "You are a helpful assistant."
POSITIONS = ["response_first", "response_last"]
MAX_PROMPT_TOKENS = 2048
MAX_RESPONSE_CHARS = 1500

slug = MODEL_CONFIGS[MODEL_KEY].slug
# Drive only holds the final directions.npz + manifest.json.
OUT_DIR = DRIVE_ROOT / slug / "exp08_directions"
OUT_DIR.mkdir(parents=True, exist_ok=True)
# Activation cache lives on the local Colab disk only — ephemeral, not synced.
CACHE_DIR = Path("/content/exp08_cache") / slug
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("OUT_DIR  (Drive, final only) =", OUT_DIR)
print("CACHE_DIR (local Colab disk) =", CACHE_DIR)
print(f"sample {N_ROWS} rows → up to {2*N_ROWS} items → up to {4*N_ROWS} forward passes")

In [ ]:
# Build the item set. Each source row yields up to 2 items:
#   item A: instruction=s_instruction, response=s_aligned_response
#   item B: instruction=u_instruction, response=u_aligned_response
# Each item is then traced under both S-arm and U-arm conditions.

@dataclass
class Item:
    item_id: str          # row_idx + which
    row_idx: int
    which: str            # 's' or 'u'
    instruction: str
    followup: str
    response: str         # shared prefill for both S and U arms
    conflict_axis: str
    macro_axis: str

def load_items(source_path: Path, n_rows: int, seed: int) -> list[Item]:
    with open(source_path) as f:
        rows = [json.loads(l) for l in f]
    rng = random.Random(seed)
    rng.shuffle(rows)
    items: list[Item] = []
    for ri, row in enumerate(rows[:n_rows]):
        followup = row.get("original_prompt") or "Hello, how can you help me?"
        for which, instr_key, resp_key in [("s", "s_instruction", "s_aligned_response"),
                                           ("u", "u_instruction", "u_aligned_response")]:
            instr = row.get(instr_key)
            resp  = row.get(resp_key)
            if not instr or not resp:
                continue
            items.append(Item(
                item_id=f"{ri:05d}_{which}",
                row_idx=ri,
                which=which,
                instruction=instr.strip(),
                followup=followup.strip(),
                response=resp.strip()[:MAX_RESPONSE_CHARS],
                conflict_axis=row.get("conflict_axis", ""),
                macro_axis=row.get("macro_axis", ""),
            ))
    return items

items = load_items(SOURCE_PATH, N_ROWS, SEED)
print(f"built {len(items)} items")
print("axis counts:")
for k, c in Counter(it.macro_axis for it in items).most_common():
    print(f"  {k!r:24s} {c}")

In [ ]:
# Load model.
loaded = load_model(MODEL_KEY)
tok = loaded.tokenizer
n_layers = loaded.n_layers
d_model = loaded.d_model
supports_thinking = getattr(loaded.template, '_supports_enable_thinking', False)
print(f'model={MODEL_KEY}  device={loaded.device}  n_layers={n_layers}  d_model={d_model}')
print(f'supports_enable_thinking={supports_thinking}')

In [ ]:
# Build a matched-baseline (system + user + assistant) bundle and return
# (full_input_ids, prompt_len, full_len). The response is prefilled as the
# assistant turn; we capture activations at response_first / response_last.

def build_bundle(instruction: str, followup: str, response: str, condition: str):
    if condition == 'S':
        sys_content = f'{BASELINE_SYSTEM}\n{instruction}'
        user_content = followup
    else:  # 'U'
        sys_content = BASELINE_SYSTEM
        user_content = f'{instruction}\n\n{followup}'
    extra = {'enable_thinking': False} if supports_thinking else {}
    msgs_prompt = [
        {'role': 'system', 'content': sys_content},
        {'role': 'user',   'content': user_content},
    ]
    msgs_full = msgs_prompt + [{'role': 'assistant', 'content': response}]

    def _flat(enc):
        if hasattr(enc, 'input_ids'): enc = enc.input_ids
        elif isinstance(enc, dict):   enc = enc['input_ids']
        if hasattr(enc, 'tolist'):    enc = enc.tolist()
        if isinstance(enc, list) and enc and isinstance(enc[0], list): enc = enc[0]
        return [int(x) for x in enc]

    prompt_ids = _flat(tok.apply_chat_template(msgs_prompt, tokenize=True,
                                               add_generation_prompt=True, **extra))
    full_ids   = _flat(tok.apply_chat_template(msgs_full, tokenize=True,
                                               add_generation_prompt=False, **extra))
    response_start = min(len(prompt_ids), len(full_ids))
    response_end   = len(full_ids)
    if response_end <= response_start:
        response_start = max(0, response_end - 1)
    return full_ids, response_start, response_end

In [ ]:
# Forward-pass + per-layer hook to extract residuals at response_first / last.
# Returns array shape (n_positions=2, n_layers, d_model).

def trace_one(input_ids: list[int], resp_start: int, resp_end: int) -> np.ndarray:
    out = np.zeros((len(POSITIONS), n_layers, d_model), dtype=np.float32)
    span_first = resp_start
    span_last  = max(resp_start, resp_end - 1)
    captured: dict[int, np.ndarray] = {}
    handles = []

    def _make_hook(li: int):
        def _hook(_m, _inp, o):
            h = o[0] if isinstance(o, tuple) else o
            captured[li] = h.detach().float().cpu().numpy()[0]   # (T, d)
        return _hook

    for li in range(n_layers):
        handles.append(loaded.layer_module(li).register_forward_hook(_make_hook(li)))
    try:
        ids_t = torch.tensor([input_ids], device=loaded.device)
        with torch.no_grad():
            loaded.hf_model(ids_t, attention_mask=torch.ones_like(ids_t), use_cache=False)
    finally:
        for h in handles:
            h.remove()

    for li in range(n_layers):
        h = captured[li]
        out[0, li, :] = h[span_first]   # response_first
        out[1, li, :] = h[span_last]    # response_last
    return out

def cache_path(item: Item, condition: str) -> Path:
    return CACHE_DIR / f'{item.item_id}_{condition}.npz'

def trace_item(item: Item, condition: str) -> np.ndarray | None:
    p = cache_path(item, condition)
    if p.exists():
        try:
            with np.load(p) as d:
                return d['acts']
        except Exception:
            p.unlink(missing_ok=True)
    full_ids, rs, re_ = build_bundle(item.instruction, item.followup, item.response, condition)
    if len(full_ids) > MAX_PROMPT_TOKENS:
        return None
    acts = trace_one(full_ids, rs, re_)
    np.savez_compressed(p, acts=acts)
    return acts

In [ ]:
# Run the trace pass. Resumable — already-cached items are skipped.

skipped_long = 0
kept = 0
t0 = time.time()
for i, it in enumerate(items):
    ok_S = trace_item(it, "S")
    ok_U = trace_item(it, "U")
    if ok_S is None or ok_U is None:
        skipped_long += 1
        continue
    kept += 1
    if (i + 1) % 25 == 0 or (i + 1) == len(items):
        rate = (i + 1) / (time.time() - t0 + 1e-9)
        eta_min = (len(items) - i - 1) / rate / 60
        print(f"  [{i+1:>5}/{len(items)}]  kept={kept}  skipped_long={skipped_long}  "
              f"{rate:.2f} it/s  eta={eta_min:.1f} min")
print(f"done — kept {kept} items, skipped {skipped_long} (>{MAX_PROMPT_TOKENS} tokens)")

In [ ]:
# Stack activations into (n, n_positions, n_layers, d_model) for S and U.
S_list, U_list, kept_ids = [], [], []
for it in items:
    pS = cache_path(it, "S"); pU = cache_path(it, "U")
    if not (pS.exists() and pU.exists()):
        continue
    with np.load(pS) as d: aS = d["acts"]
    with np.load(pU) as d: aU = d["acts"]
    S_list.append(aS); U_list.append(aU); kept_ids.append(it.item_id)
S = np.stack(S_list, axis=0)  # (n, 2, n_layers, d)
U = np.stack(U_list, axis=0)
print(f"paired traces: n={S.shape[0]}  shape={S.shape}")

In [ ]:
# Fit MM, pca_diff, pca_center per (position × layer). Sign-fix so projection
# of mean_S beats mean_U (matches our 'positive = system-following' convention).

def fit_mm(h_S, h_U):
    raw = h_S.mean(0) - h_U.mean(0)
    n = float(np.linalg.norm(raw))
    unit = raw / (n + 1e-10) if n > 1e-10 else np.zeros_like(raw)
    return unit.astype(np.float32), raw.astype(np.float32)

def fit_pca_diff(h_S, h_U):
    diff = h_S - h_U
    pca = PCA(n_components=1, whiten=False).fit(diff)
    v = pca.components_[0].astype(np.float32)
    return v / (np.linalg.norm(v) + 1e-8)

def fit_pca_center(h_S, h_U):
    mid = (h_S + h_U) / 2.0
    train = np.concatenate([h_S - mid, h_U - mid], axis=0)
    pca = PCA(n_components=1, whiten=False).fit(train)
    v = pca.components_[0].astype(np.float32)
    return v / (np.linalg.norm(v) + 1e-8)

def sign_fix(v, h_S, h_U):
    if (h_S @ v).mean() < (h_U @ v).mean():
        return -v
    return v

directions: dict[str, np.ndarray] = {}
manifest = {
    'n_paired': int(S.shape[0]),
    'n_layers': int(n_layers),
    'd_model':  int(d_model),
    'positions': POSITIONS,
    'baseline_system': BASELINE_SYSTEM,
    'source_path': str(SOURCE_PATH),
    'n_rows_sampled': int(N_ROWS),
    'seed': int(SEED),
    'model_key': MODEL_KEY,
    'per_layer': {pos: {} for pos in POSITIONS},
}

for pi, pos in enumerate(POSITIONS):
    print(f'\n[fit] position={pos}')
    for li in range(n_layers):
        h_S = S[:, pi, li, :]
        h_U = U[:, pi, li, :]

        mm_unit, mm_raw = fit_mm(h_S, h_U)
        v_diff   = fit_pca_diff(h_S, h_U)
        v_center = fit_pca_center(h_S, h_U)
        v_diff   = sign_fix(v_diff,   h_S, h_U)
        v_center = sign_fix(v_center, h_S, h_U)

        directions[f'mm_dir__{pos}__layer_{li:03d}']         = mm_unit
        directions[f'mm_raw__{pos}__layer_{li:03d}']         = mm_raw
        directions[f'pca_diff_dir__{pos}__layer_{li:03d}']   = v_diff
        directions[f'pca_center_dir__{pos}__layer_{li:03d}'] = v_center

        manifest['per_layer'][pos][str(li)] = {
            'mm_natural_scale':       float(np.linalg.norm(mm_raw)),
            'cos_pca_diff_vs_mm':     float(v_diff   @ mm_unit),
            'cos_pca_center_vs_mm':   float(v_center @ mm_unit),
            'cos_pca_diff_vs_center': float(v_diff   @ v_center),
        }
        if li in (0, 5, 10, 16, 20, 25, 31):
            print(f'  L{li:02d}  ||MM_raw||={np.linalg.norm(mm_raw):.3f}  '
                  f'cos(diff,MM)={v_diff@mm_unit:+.3f}  '
                  f'cos(ctr,MM)={v_center@mm_unit:+.3f}  '
                  f'cos(diff,ctr)={v_diff@v_center:+.3f}')

print(f'\nfit complete — {len(directions)} arrays')

In [ ]:
# Save the single deliverable: directions.npz + manifest.json (to Drive).
out_npz = OUT_DIR / "directions.npz"
np.savez_compressed(out_npz, **directions)
with open(OUT_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
print(f"wrote {out_npz}  ({len(directions)} arrays)")
print(f"wrote {OUT_DIR/'manifest.json'}")
print()
print("Sample keys:")
for k in list(directions)[:6]:
    print(f"  {k}: shape={directions[k].shape}")
print("  ...")
print()
print("To use in 07_steering_authority.ipynb locally:")
print(f"  scp/cp the file from Drive to <repo>/exp08_directions/directions.npz")
print( "  then point PCA_PATH (and the MM loader) at that file.")